# 38장 · 재검색 루프 · 검색어를 고치고 두 번 안에 멈추기

재검색 루프는 첫 검색의 근거가 부족할 때 검색어를 바꿔 다시 찾습니다. 사용자가 물어본 질문은 따로 남기고, 검색 횟수를 세어 같은 경로가 끝없이 반복되는 것을 막습니다.

## 두 번째 실패에서 멈추는 분기

`after_search`는 근거 유무와 지금까지의 검색 횟수로 다음 행동을 고릅니다. 두 번째 검색도 실패한 상태를 넣어 한 번 더 찾을지, 보류하고 끝낼지 확인하세요.

In [ ]:
def after_search(has_evidence, attempts):
    if has_evidence:
        return "answer"
    return "search_again" if attempts < 2 else "stop"

print(after_search(False, 2))

stop


두 번 검색했는데 근거가 없어 stop이 나옵니다. attempts를 1로 바꾸면 search_again이 됩니다.

## 응용 실습 · 검색어를 바꿔 다시 찾기

짝 활동에는 [실습 수첩](pre-mid/documents/graph/workshop.md)을 사용합니다. 과제 아래에서 풀이를 펼칠 수 있습니다.

### 인용에 쓸 수 있었던 후보 다시 열기

37장에서 통과한 문장과 보류한 문장을 구별합니다. 이 결과를 첫 검색으로 재생한 뒤, 재검색을 한 번만 허용하면 어느 종료 노드에 닿는지 확인하세요. 통합 ZIP을 모두 풀고 실행하세요. 내 파일이 없으면 출력에 표시된 교재 예시로 시작합니다.

In [ ]:
from pathlib import Path
import json
import sys

root = next(folder for folder in (Path.cwd(), *Path.cwd().parents)
            if (folder / "pre-mid/documents/workflow").is_dir())
sys.path.insert(0, str(root / "pre-mid/documents/workflow"))
from workbook_files import read_previous, save_record

previous = read_previous(root, 37)
print(previous["status"], previous["quote_id"], previous["quote"])

내 작업 · chapter37.json
인용 표시 recorder-a-1.1.pdf#p2 한 번에 길게 녹음하면 45분마다 새 파일이 열립니다. 파일이 바뀌어도 녹음은 계속됩니다.


## 재검색할 query와 사용자 question 분리하기

`rewrite` 노드는 검색용 `query`를 바꾸고 방문 기록 `path`에 `rewrite`를 추가합니다. 다음 검색은 그 값을 쓰고, 답변 노드는 여전히 처음의 `question`을 읽습니다. `attempt`는 실제로 검색한 뒤 올라가므로 두 번째 검색까지 실패하면 `hold`로 끝낼 수 있습니다.

사용자가 쓴 표현은 `compose` 노드까지 유지합니다. 검색어를 “표시등 설정”으로 바꾸면 자료에서 쓰는 용어에 가까워지지만, “녹음 중”이 빠지면 시작 전과 도중의 차이가 사라집니다. 재작성에서는 대상, 판본, 시각, 요청 행동 중 무엇을 보존할지 원문에 밑줄을 긋고 시작하세요.

3부에서 쓰던 세 PDF와 공용 검색기를 같은 폴더에서 엽니다. 교재 ZIP 안의 `pre-mid/documents/rag`, `graph`, `sources`가 함께 있어야 합니다. 실제 검색 셀은 3부 Chroma 색인·OpenAI 임베딩과 API 키를 사용합니다. 처음 실행한다면 노트북 셀에서 `%pip install "langgraph>=1,<2" "langchain-openai>=1,<2" "chromadb>=1,<2" pypdf openai python-dotenv streamlit`를 실행하고 커널을 다시 시작하세요. PDF 직접 읽기와 고정 입력 경로 실험은 키 없이 실행됩니다.

```mermaid
flowchart TD
  R[검색] --> J[근거 판정]
  J -->|충분| A[답변과 출처]
  J -->|첫 실패| W[검색어 재작성]
  W --> R
  J -->|두 번째 실패| H[답변 보류]
```

In [1]:
from pathlib import Path
import sys

# 통합 ZIP과 저장소에서 같은 자료 폴더를 찾습니다.
docs = next(p / "pre-mid" / "documents" for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pre-mid" / "documents" / "graph").is_dir())
sys.path.insert(0, str(docs / "graph"))
sys.path.insert(0, str(docs / "rag"))
from rag_lab import MANUALS, read_pages

# 세 설명서의 모든 페이지를 실제 PDF에서 읽습니다.
pages = [page for path in MANUALS for page in read_pages(path)]
by_id = {page["id"]: page for page in pages}
from recorder_graph import assemble_graph, ask, next_step, validate_quotes

for attempt, evidence in [(1, []), (2, []), (2, [pages[0]])]:
    print(attempt, bool(evidence), next_step({"attempt": attempt, "evidence": evidence}))

1 False rewrite
2 False hold
2 True compose


답할 수 있는 근거가 있으면 두 번째 검색이어도 답합니다. 근거가 없을 때만 남은 검색 횟수를 봅니다. 검색어가 바뀌었다는 사실은 종료 조건이 될 수 없습니다.

### 실습 · 두 번째 검색에서 근거를 찾으면 어디로 갈까요?

같은 두 번째 검색이라도 근거가 없을 때와 한 문장을 찾았을 때 목적지가 다릅니다. 수업 함수 next_step에 두 상태를 보내고 반환된 노드 이름을 비교하세요.

In [ ]:
from recorder_graph import next_step
state = {"attempt": 2, "evidence": []}
# 첫 목적지를 읽고 evidence만 바꿔 다시 확인하세요.

<details><summary>정답 코드와 실행 결과</summary><pre><code>from recorder_graph import next_step
state = {&quot;attempt&quot;: 2, &quot;evidence&quot;: []}
print(next_step(state))
state[&quot;evidence&quot;] = [{&quot;quote&quot;: &quot;오후 네 시&quot;}]
print(next_step(state))</code></pre><pre>hold
compose</pre><p>두 번째 실패는 hold입니다. 두 번째라도 근거가 있으면 compose가 먼저 선택됩니다. 이 예제는 분기 함수의 반환값을 확인하므로 검색기나 모델을 부르지 않습니다.</p></details>

## 입력 후보를 고정해 세 종료 경로 재현하기

경로에 결함이 있는지 확인하려면 원하는 상황을 반복해서 만들어야 합니다. 실제 PDF의 문단 두 개를 읽되 검색 결과 순서는 우리가 고정합니다. 이 실험은 경로 검사용이며 실제 Chroma의 순위를 나타내지 않습니다. 학생이 먼저 각 행에서 몇 번 검색할지 적은 뒤 실행하세요.

```mermaid
flowchart TD
start["START · 질문"] --> retrieve["retrieve · 후보 검색"]
retrieve --> judge["judge · 적용 판본과 근거 검사"]
judge -->|근거 있음| compose["compose · 답과 출처 작성"]
judge -->|근거 없음, 첫 검색| rewrite["rewrite · 검색어 수정"]
rewrite --> retrieve
judge -->|근거 없음, 두 번째 검색| hold["hold · 응답 보류"]
compose --> finish["END"]
hold --> finish
```

In [2]:
from recorder_graph import assemble_graph, ask, validate_quotes

right = by_id["recorder-a-1.1.pdf#p1"]
wrong = by_id["recorder-b-1.0.pdf#p1"]
light_question = "5월 A형은 녹음하는 동안 앞쪽 빨간불을 끌 수 있나요?"
def fixed_rewrite(question, model, edition):
    # 일상어 한 단어만 설명서 용어로 바꾸고 원래 행동과 시점은 남깁니다.
    return f"{model}형 {edition}판 | " + question.replace("빨간불", "표시등")

def fixed_trial(first, second, quote):
    def search(query):
        return second if "판 | " in query else first
    def assess(question, candidates):
        # 실습자가 표시한 문구를 후보에서 검증합니다. 모델 판정은 아래 실제 검색 셀에서 사용합니다.
        if not quote:
            return [], "직접 답하는 문장 없음"
        evidence = validate_quotes(candidates, [{"id": right["id"], "quote": quote}])
        return evidence, "인용문 확인" if evidence else "인용 후보 없음"
    return assemble_graph(search, assess, lambda q, docs: docs[0]["quote"], fixed_rewrite)

light_quote = "녹음하는 동안 표시등을 끄는 설정은 없습니다."
for label, first, second in [("직접", [right], []), ("재검색", [wrong], [right]), ("보류", [wrong], [wrong])]:
    result = ask(fixed_trial(first, second, light_quote), light_question, "A", "1.1")
    print(label, result["attempt"], result["path"], result["status"])

직접 1 ['retrieve', 'judge', 'compose'] answer
재검색 2 ['retrieve', 'judge', 'rewrite', 'retrieve', 'judge', 'compose'] answer
보류 2 ['retrieve', 'judge', 'rewrite', 'retrieve', 'judge', 'hold'] abstain


세 번째 경우에도 첫 검색은 일어났고 두 번째까지 했지만 선택한 판본의 근거가 없었습니다. 답변 노드는 한 번도 실행되지 않습니다. 시도 수, 방문 순서, 종료 상태를 함께 읽으면 “두 번째 답이 없었다”의 이유를 설명할 수 있습니다.

## 재검색이 실제로 무슨 차이를 만들었는가

초회 검색 후보를 새 검색 결과로 덮어쓰면 첫 검색이 실패한 이유를 복기할 수 없습니다. searches에는 매번 검색어와 후보 목록 한 벌을 덧붙입니다. documents는 최근 후보만 가리킵니다.

```mermaid
flowchart TD
start["START · 질문"] --> retrieve["retrieve · 후보 검색"]
retrieve --> judge["judge · 적용 판본과 근거 검사"]
judge -->|근거 있음| compose["compose · 답과 출처 작성"]
judge -->|근거 없음, 첫 검색| rewrite["rewrite · 검색어 수정"]
rewrite --> retrieve
judge -->|근거 없음, 두 번째 검색| hold["hold · 응답 보류"]
compose --> finish["END"]
hold --> finish
```

In [3]:
recovered = ask(fixed_trial([wrong], [right], light_quote), light_question, "A", "1.1")
print("사용자가 물은 말", recovered["question"])
for record in recovered["searches"]:
    print(record["query"], [p["id"] for p in record["documents"]])
print(recovered["citations"][0])

사용자가 물은 말 5월 A형은 녹음하는 동안 앞쪽 빨간불을 끌 수 있나요?
5월 A형은 녹음하는 동안 앞쪽 빨간불을 끌 수 있나요? ['recorder-b-1.0.pdf#p1']
A형 1.1판 | 5월 A형은 녹음하는 동안 앞쪽 표시등을 끌 수 있나요? ['recorder-a-1.1.pdf#p1']
{'id': 'recorder-a-1.1.pdf#p1', 'source': 'recorder-a-1.1.pdf', 'page': 1, 'quote': '녹음하는 동안 표시등을 끄는 설정은 없습니다.'}


첫 검색의 B형과 둘째 검색의 A형은 서로 다른 ID입니다. 인용 목록에 적힌 파일과 쪽은 최종 근거에서 가져옵니다. 첫 후보를 무조건 인용하면 경로는 맞아도 근거는 틀립니다.

## 조건부 간선으로 재검색 횟수 제한하기

34장에서 길이 조건으로 재작성을 선택했던 것처럼, 여기서는 evidence와 attempt가 다음 움직임을 정합니다. 아래에서 검색·판정·재작성 노드를 작은 입력으로 직접 연결해 봅니다. 이번 실험의 검색기는 실제 서비스 대신 어떤 검색어가 들어왔는지에 따라 PDF 후보를 건넵니다. 조건부 간선에는 다음으로 갈 수 있는 노드 이름을 명시합니다.

`choose_node(state)`는 다음 노드의 이름만 반환합니다. 우선 근거가 있으면 `compose`, 근거가 없고 검색을 한 번만 했다면 `rewrite`, 두 번 했다면 `hold`입니다. 횟수는 실제 검색을 한 `search_node`에서만 증가합니다. 초기값이 0일 때 첫 실패 직후의 상태에는 1이 들어 있어야 재검색 기회가 정확히 한 번 남습니다.

```mermaid
flowchart TD
start["START · 질문"] --> search["search · 후보 검색"]
search --> judge["judge · 적용 판본과 근거 검사"]
judge -->|근거 있음| compose["compose · 답과 출처 작성"]
judge -->|근거 없음, 첫 검색| rewrite["rewrite · 검색어 수정"]
rewrite --> search
judge -->|근거 없음, 두 번째 검색| hold["hold · 응답 보류"]
compose --> finish["END"]
hold --> finish
```

In [4]:
from langgraph.graph import START, END, StateGraph
from recorder_graph import RecorderState, matching_pages

def search_node(state):
    candidates = [right] if "표시등" in state["query"] else [wrong]
    return {"documents": candidates, "attempt": state["attempt"] + 1}

def judge_node(state):
    selected = matching_pages(state["documents"], state["model"], state["edition"])
    evidence = validate_quotes(selected, [{"id": right["id"], "quote": light_quote}])
    return {"evidence": evidence}

def choose_node(state):
    if state["evidence"]:
        return "compose"
    return "rewrite" if state["attempt"] < 2 else "hold"

plan = StateGraph(RecorderState)
plan.add_node("search", search_node)
plan.add_node("judge", judge_node)
plan.add_node("rewrite", lambda s: {"query": fixed_rewrite(s["question"], s["model"], s["edition"])})
plan.add_node("compose", lambda s: {"answer": s["evidence"][0]["quote"]})
plan.add_node("hold", lambda s: {"answer": "근거 확인 필요"})
plan.add_edge(START, "search")
plan.add_edge("search", "judge")
plan.add_conditional_edges("judge", choose_node, ["compose", "rewrite", "hold"])
plan.add_edge("rewrite", "search")
plan.add_edge("compose", END)
plan.add_edge("hold", END)
practice_graph = plan.compile()
initial = {"question": light_question, "query": light_question, "model": "A", "edition": "1.1", "attempt": 0}
for update in practice_graph.stream(initial, stream_mode="updates"):
    print(next(iter(update)))

search
judge
rewrite
search
judge
compose


rewrite 다음에 search가 다시 방문돼야 새 표현으로 문단을 찾습니다. 첫 후보와 둘째 후보를 같은 페이지로 고정해 놓고 종이에서 방문 순서를 다시 그려 보세요. 두 번째에도 근거가 없으면 choose_node는 hold를 반환하고 끝납니다. 실제 그래프에는 앞의 배선에 검색 기록 저장과 모델 판정을 붙였습니다.

## 실제 검색기·판정 모델·생성 모델 연결

make_graph는 3부의 retrieve를 검색 노드에서 부릅니다. 기종·판본이 일치한 후보에 대해서만 판정 모델에 직접 인용을 요청합니다. 부족하면 한 번 검색어 재작성 후 다시 찾습니다. 실제 검색 순서는 고정 실험과 다를 수 있으므로 어느 경로가 나왔든 근거를 열어 확인합니다.

In [5]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# 8장에서 준비한 .env를 읽습니다. 키는 출력하지 않습니다.
load_dotenv()
use_live = bool(os.getenv("OPENAI_API_KEY"))
model_name = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
if use_live:
    client = OpenAI()

```mermaid
flowchart TD
start["START · 질문"] --> retrieve["retrieve · 후보 검색"]
retrieve --> judge["judge · 적용 판본과 근거 검사"]
judge -->|근거 있음| compose["compose · 답과 출처 작성"]
judge -->|근거 없음, 첫 검색| rewrite["rewrite · 검색어 수정"]
rewrite --> retrieve
judge -->|근거 없음, 두 번째 검색| hold["hold · 응답 보류"]
compose --> finish["END"]
hold --> finish
```

In [6]:
if use_live:
    from rag_lab import open_index
    from recorder_graph import make_graph
    live_graph = make_graph(open_index(), k=2)
    outcome = ask(live_graph, light_question, "A", "1.1")
    for search in outcome["searches"]:
        print(search["query"], [p["id"] for p in search["documents"]])
    print(outcome["path"], outcome["status"], outcome["reason"])
    print(outcome["answer"], outcome["citations"])
else:
    print("실제 검색 경로는 API 키를 준비한 뒤 실행하세요.")

실제 검색 경로는 API 키를 준비한 뒤 실행하세요.


검색 결과는 사용한 설정과 모델 응답에 따라 달라집니다. 참조 답은 A형 1.1판 1쪽에서 녹음 중 표시등을 끌 수 없다는 문장을 찾는 것입니다. 결과가 다르면 재검색 횟수를 늘리기 전에 올바른 쪽이 후보에 있었는지 먼저 확인하세요.

## 실습 · 두 번째에도 없을 때 남길 정보

처음과 두 번째 모두 B형만 나온 경우를 재현합니다. 학생이 고객에게 보낸 말과 개발자에게 남길 진단 정보를 따로 쓰세요. 보류한 결과에 인용이 실수로 남지 않았는지도 확인합니다.

```mermaid
flowchart TD
start["START · 질문"] --> retrieve["retrieve · 후보 검색"]
retrieve --> judge["judge · 적용 판본과 근거 검사"]
judge -->|근거 있음| compose["compose · 답과 출처 작성"]
judge -->|근거 없음, 첫 검색| rewrite["rewrite · 검색어 수정"]
rewrite --> retrieve
judge -->|근거 없음, 두 번째 검색| hold["hold · 응답 보류"]
compose --> finish["END"]
hold --> finish
```

In [7]:
attempts_to_test = [[wrong], [wrong]]
# 두 번 모두 이 후보를 주는 fixed_trial로 ask를 실행하세요. answer·citations와 두 검색어를 확인하세요.

<details><summary>모범 예시와 풀이</summary><pre><code>r = ask(fixed_trial([wrong], [wrong], light_quote), light_question, &quot;A&quot;, &quot;1.1&quot;)
print(r[&quot;answer&quot;], r[&quot;citations&quot;])
print(r[&quot;attempt&quot;], r[&quot;reason&quot;], [x[&quot;query&quot;] for x in r[&quot;searches&quot;]])</code></pre><p>고객에게는 요청한 판본에서 근거를 확인하지 못했다고 말합니다. 개발자는 두 검색어와 후보를 보고 검색 조건을 검토합니다. “A형에 그 기능이 없다”는 단정은 이 기록만으로 할 수 없습니다.</p></details>

## 실습 · 질문을 다른 기능으로 바꿔 실패의 성격 읽기

“A형은 녹음 중 외부 소음을 자동 제거하나요?”를 새로 받았습니다. A형 1.1판이 두 번 모두 검색됐지만 문서에는 관련 기능 설명이 없습니다. 앞의 B형 사례와 종료 상태가 같아도 보완할 방법이 어떻게 다른지 확인하세요.

```mermaid
flowchart TD
start["START · 질문"] --> retrieve["retrieve · 후보 검색"]
retrieve --> judge["judge · 적용 판본과 근거 검사"]
judge -->|근거 있음| compose["compose · 답과 출처 작성"]
judge -->|근거 없음, 첫 검색| rewrite["rewrite · 검색어 수정"]
rewrite --> retrieve
judge -->|근거 없음, 두 번째 검색| hold["hold · 응답 보류"]
compose --> finish["END"]
hold --> finish
```

In [8]:
noise = "A형은 녹음 중 외부 소음을 자동 제거하나요?"
# 두 검색 모두 [right]를 주고 인용은 빈 문자열로 둔 fixed_trial을 만드세요. 답의 status와 citations를 읽으세요.

<details><summary>모범 예시와 풀이</summary><pre><code>noise = &quot;A형은 녹음 중 외부 소음을 자동 제거하나요?&quot;
r = ask(fixed_trial([right], [right], &quot;&quot;), noise, &quot;A&quot;, &quot;1.1&quot;)
print(r[&quot;status&quot;], r[&quot;reason&quot;], r[&quot;citations&quot;])</code></pre><p>여기서는 기종이 맞아도 질문에 답할 문장이 없습니다. 자료 자체에 설명이 없는 경우에는 검색 횟수를 늘려도 해결이 보장되지 않습니다. 제조사 자료를 추가하거나 확인이 필요함을 알립니다.</p></details>

## 전이 과제 · 그럴듯한 재검색어가 질문을 바꿨을 때

원래 질문은 “B형으로 이미 녹음 중인데 빨간불을 지금 감추려면?”입니다. 학생 A는 “B형 표시등 숨김 설정”, 학생 B는 “B형 1.0판 이미 녹음 중 표시등 설정 변경”으로 검색합니다. 둘 다 같은 1쪽을 찾을 수 있지만 A의 검색어만 읽고 답을 만들면 녹음 *전*에 숨김을 선택한다는 안내로 끝나기 쉽습니다.

두 검색어가 보존하거나 빠뜨린 조건을 표시하세요. B형 1.0판 1쪽에서 “이미 녹음 중”의 답과 “시작 전”의 답을 각각 찾아 다른 문장에 밑줄을 긋습니다. 재검색어가 원래 질문을 대체하지 않게 유지할 상태 키 두 개를 적고, 실제 그래프에 넣어 최종 답을 확인하세요.

In [9]:
page_id = "recorder-b-1.0.pdf#p1"
# by_id에서 이 쪽을 읽고 녹음 전과 녹음 중의 설정 문장을 찾으세요. 새 검색어에서도 질문의 시점이 유지됐나요?

<details><summary>모범 예시와 풀이</summary><pre><code>page = by_id[&quot;recorder-b-1.0.pdf#p1&quot;]
for text in [&quot;이미 녹음 중이면 표시등 설정을 바꿀 수 없습니다.&quot;, &quot;녹음 전에 설정에서 표시등을 ‘숨김’으로 바꾸세요.&quot;]:
    print(text, text in &quot; &quot;.join(page[&quot;text&quot;].split()))
if use_live:
    question = &quot;5월 B형으로 이미 녹음 중인데 빨간불을 지금 감추려면?&quot;
    r = ask(live_graph, question, &quot;B&quot;, &quot;1.0&quot;)
    print(r[&quot;question&quot;], r[&quot;query&quot;], r[&quot;answer&quot;])</code></pre><p>원래 질문은 question, 검색용 문장은 query입니다. 이미 녹음 중이면 그대로 설정을 바꿀 수 없습니다. 녹음을 끝낸 뒤 숨김을 선택하고 새 녹음을 시작하는 절차가 필요합니다. 같은 페이지를 인용해도 어느 시점의 행동을 답했는지 읽어야 합니다.</p></details>

## 앞 장의 후보로 재검색과 종료 확인하기

첫 검색에는 37장의 후보를, 두 번째 검색에는 실제 PDF에서 읽어 둔 문단을 제공합니다. 경로를 비교하려고 검색 결과를 고정한 실험입니다. 자신의 인용문이 없으면 두 번째 검색 뒤 hold로 끝나는지 확인하세요.

```mermaid
flowchart TD
start["START · 질문"] --> retrieve["retrieve · 후보 검색"]
retrieve --> judge["judge · 적용 판본과 근거 검사"]
judge -->|근거 있음| compose["compose · 답과 출처 작성"]
judge -->|근거 없음, 첫 검색| rewrite["rewrite · 검색어 수정"]
rewrite --> retrieve
judge -->|근거 없음, 두 번째 검색| hold["hold · 응답 보류"]
compose --> finish["END"]
hold --> finish
```

In [ ]:
def replay_search(query):
    return previous["documents"] if query == previous["question"] else pages

def replay_judge(question, candidates):
    quotes = [{"id": previous["quote_id"], "quote": previous["quote"]}]
    evidence = validate_quotes(candidates, quotes)
    return evidence, "앞 장의 인용을 원문과 대조"

replay_graph = assemble_graph(replay_search, replay_judge, lambda q, ev: ev[0]["quote"], fixed_rewrite)
replay = ask(replay_graph, previous["question"], previous["model"], previous["edition"])
print(" → ".join(replay["path"]), replay["status"])
save_record(root, 38, {"question": replay["question"], "model": replay["model"], "edition": replay["edition"],
                      "answer": replay["answer"], "status": replay["status"], "path": replay["path"],
                      "reason": replay["reason"], "citations": replay["citations"], "index_version": None,
                      "expected_page": previous["reference_page_id"], "evidence_mode": "경로 연습",
                      "searches": [{"query": s["query"], "ids": [p["id"] for p in s["documents"]]}
                                   for s in replay["searches"]]})

retrieve → judge → compose answer
my-course/workflow/chapter38.json


다음 장에서는 이 결과를 사용자가 읽을 부분과 운영자가 펼쳐 볼 부분으로 나눠 표시합니다.